# EEG Dataset Unit Check & Loader Test
KL 2026-01-06

In [2]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
from loader import load_dataset, split_dataset, SplitStrategy
from hdf5_io import HDF5Reader
import numpy as np

In [3]:
def detect_unit(data: np.ndarray) -> tuple[str, float]:
    """Detect data unit based on amplitude range."""
    max_amp = np.abs(data).max()
    if max_amp > 1e-2:  # > 0.01, likely µV
        return "µV", max_amp
    return "V", max_amp


def check_dataset_unit(data_dir: str) -> dict:
    """Check unit and amplitude statistics for a dataset."""
    h5_files = sorted(Path(data_dir).glob("*_*.h5"))
    print(f"Found {len(h5_files)} HDF5 files in {data_dir}")
    if not h5_files:
        return {"error": "No HDF5 files found"}

    stats = {
        "dataset_name": None,
        "num_files": len(h5_files),
        "num_labels": None,
        "category_list": None,
        "detected_unit": None,
        "max_amplitude": 0,
        "min_amplitude": float('inf'),
        "mean_amplitude": [],
        "samples_checked": 0,
        "amplitude_violations": 0,
    }

    for h5_file in h5_files[:3]:
        print(f"Checking file: {h5_file}")
        with HDF5Reader(str(h5_file)) as reader:
            subj = reader.subject_attrs
            if stats["dataset_name"] is None:
                stats["dataset_name"] = subj.dataset_name
                stats["num_labels"] = getattr(subj, 'num_labels', None)
                stats["category_list"] = getattr(subj, 'category_list', None)

            trial_names = reader.get_trial_names()[:3]
            for trial_name in trial_names:
                seg_names = reader.get_segment_names(trial_name)[:5]
                # `subj = reader.subject_attrs` is assigning the attribute `subject_attrs` of the `reader`
                # object to the variable `subj`. This line of code is retrieving the subject attributes from
                # the HDF5 file being read by the `reader` object. These attributes typically contain
                # information about the dataset, task type, subject ID, sampling rate, montage, channels, and
                # other relevant metadata related to the EEG data being processed.
                for seg_name in seg_names:
                    seg = reader.get_segment(trial_name, seg_name)
                    data = seg.data
                    max_amp = np.abs(data).max()
                    stats["max_amplitude"] = max(stats["max_amplitude"], max_amp)
                    stats["min_amplitude"] = min(stats["min_amplitude"], max_amp)
                    stats["mean_amplitude"].append(np.abs(data).mean())
                    stats["samples_checked"] += 1
                    if max_amp > 600:
                        stats["amplitude_violations"] += 1

    stats["detected_unit"], _ = detect_unit(np.array([stats["max_amplitude"]]))
    stats["mean_amplitude"] = np.mean(stats["mean_amplitude"]) if stats["mean_amplitude"] else 0
    return stats

## Single Dataset Report

In [4]:
# Change this to your dataset path
data_dir = "/mnt/dataset2/Processed_datasets/EEG_Bench/Benchmark"

In [5]:
stats = check_dataset_unit(data_dir)

print(f"Dataset: {stats['dataset_name']}")
print(f"Files: {stats['num_files']}")
print(f"Samples checked: {stats['samples_checked']}")
print()
print("--- Label Info ---")
print(f"Num labels: {stats['num_labels']}")
print(f"Categories: {stats['category_list']}")
print()
print("--- Unit Analysis ---")
print(f"Detected unit: {stats['detected_unit']}")
print(f"Max amplitude: {stats['max_amplitude']:.4e}")
print(f"Min amplitude: {stats['min_amplitude']:.4e}")
print(f"Mean amplitude: {stats['mean_amplitude']:.4e}")
print()
print("--- Validation ---")
is_uv = stats['detected_unit'] == 'µV'
has_labels = stats['num_labels'] is not None and stats['num_labels'] > 0
has_categories = stats['category_list'] is not None and len(stats['category_list']) > 0
print(f"Unit is µV: {'✓' if is_uv else '✗ (NEEDS UPDATE)'}")
print(f"Has num_labels: {'✓' if has_labels else '✗ (NEEDS UPDATE)'}")
print(f"Has category_list: {'✓' if has_categories else '✗ (NEEDS UPDATE)'}")
if is_uv:
    print(f"Amplitude violations (>600 µV): {stats['amplitude_violations']}/{stats['samples_checked']}")

Found 35 HDF5 files in /mnt/dataset2/Processed_datasets/EEG_Bench/Benchmark
Checking file: /mnt/dataset2/Processed_datasets/EEG_Bench/Benchmark/sub_1.h5
Checking file: /mnt/dataset2/Processed_datasets/EEG_Bench/Benchmark/sub_10.h5
Checking file: /mnt/dataset2/Processed_datasets/EEG_Bench/Benchmark/sub_11.h5
Dataset: Benchmark_40Class
Files: 35
Samples checked: 45

--- Label Info ---
Num labels: 40
Categories: ['freqs_label.csv']

--- Unit Analysis ---
Detected unit: µV
Max amplitude: 4.5525e+02
Min amplitude: 2.6470e+01
Mean amplitude: 7.1469e+00

--- Validation ---
Unit is µV: ✓
Has num_labels: ✓
Has category_list: ✓
Amplitude violations (>600 µV): 0/45


In [2]:
import mne
file_path = '/mnt/dataset2/Datasets/Efficient dual-frequency SSVEP BCI system/Data_of_1_Target/Data of 1Target/Subject 02.cnt'
raw = mne.io.read_raw_cnt(file_path, preload=True)

df = raw.annotations.to_data_frame()
# raw.annotations.count()
df = df.rename(columns={'onset': 'time_sec'})      # onset 
print(df[['time_sec', 'duration', 'description']].head(400))

Reading 0 ... 529719  =      0.000 ...   529.719 secs...


/tmp/ipykernel_13819/4037277588.py:3: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(file_path, preload=True)


                  time_sec  duration description
0  1970-01-01 00:00:26.062       0.0           3
1  1970-01-01 00:00:31.075       0.0         253
2  1970-01-01 00:00:36.110       0.0           7
3  1970-01-01 00:00:41.125       0.0         253
4  1970-01-01 00:00:46.142       0.0           5
..                     ...       ...         ...
75 1970-01-01 00:08:23.520       0.0         253
76 1970-01-01 00:08:28.554       0.0           4
77 1970-01-01 00:08:33.570       0.0         253
78 1970-01-01 00:08:38.603       0.0           2
79 1970-01-01 00:08:43.620       0.0         253

[80 rows x 3 columns]


## Multi-Dataset Report

In [ ]:
# Base directory containing all datasets
base_dir = "/mnt/dataset2/Processed_datasets/EEG_Bench/Thing_EEG2_2Class"

In [ ]:
base_path = Path(base_dir)
datasets = sorted([d for d in base_path.iterdir() if d.is_dir()])
print(datasets)
results = []


for dataset_dir in datasets:
    h5_files = list(dataset_dir.glob("sub_*.h5"))
    if not h5_files:
        continue
    stats = check_dataset_unit(str(dataset_dir))
    stats["path"] = dataset_dir.name
    results.append(stats)

# Print summary table
print(f"{'Dataset':<20} {'Unit':<6} {'Max Amp':<12} {'Labels':<8} {'Categories':<10}")
print("-" * 70)

for r in results:
    if "error" in r:
        print(f"{r.get('path', 'Unknown'):<20} ERROR")
        continue
    unit_ok = "✓" if r['detected_unit'] == 'µV' else "✗"
    labels_ok = "✓" if r['num_labels'] and r['num_labels'] > 0 else "✗"
    cats_ok = "✓" if r['category_list'] and len(r['category_list']) > 0 else "✗"
    print(f"{r['path']:<20} {r['detected_unit']:<6} {r['max_amplitude']:<12.2e} {labels_ok:<8} {cats_ok:<10}")

print()
needs_update = [r['path'] for r in results if 'error' not in r and r['detected_unit'] != 'µV']
if needs_update:
    print(f"Datasets needing unit update: {needs_update}")
else:
    print("All datasets are in µV ✓")

[PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/AD65'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/BCIC2A'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/CineBrain'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/Daily_Movie'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/EmoEEG-MC_emo'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/FACED_emo'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/HFO'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/SEED'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/SEEDIV'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/SEEDV')]


ValueError: invalid literal for int() with base 10: 'None'

## Batch Loading Test

In [ ]:
loader = load_dataset(data_dir, batch_size=32, num_workers=0)
print(f"Total samples: {len(loader.dataset)}")

for i, batch in enumerate(loader):
    if i >= 3:
        break
    unit, max_amp = detect_unit(batch['data'].numpy())
    print(f"Batch {i}: shape={batch['data'].shape}, unit={unit}, max_amp={max_amp:.2e}, labels={batch['label'].squeeze().tolist()[:5]}...")

## HDF5 Metadata Inspection

In [ ]:
h5_files = sorted(Path(data_dir).glob("sub_*.h5"))
if h5_files:
    with HDF5Reader(str(h5_files[0])) as reader:
        subj = reader.subject_attrs
        print(f"Dataset name: {subj.dataset_name}")
        print(f"Task type: {subj.task_type}")
        print(f"Downstream task: {subj.downstream_task_type}")
        print(f"Subject ID: {subj.subject_id}")
        print(f"Sampling rate: {subj.rsFreq} Hz")
        print(f"Montage: {subj.montage}")
        print(f"Channels ({len(subj.chn_name)}): {subj.chn_name[:5]}...")
        print(f"Num labels: {getattr(subj, 'num_labels', 'N/A')}")
        print(f"Categories: {getattr(subj, 'category_list', 'N/A')}")
        print(f"Num trials: {len(reader.get_trial_names())}")
        print(f"Num segments: {len(reader)}")